# 06 · Landsat 8 (imágenes) y MapBiomas (indicadores), 2013 → actualidad  ·  v2

> **▶ Ejecuta las celdas en orden, empezando por la primera celda de código (configuración), o usa «Run All».** Si abres este archivo y ves otra cosa, cierra y vuelve a abrir el .ipynb para que se cargue la versión guardada.

**Reparto de roles tras el rediseño del proyecto**
- **Imágenes: Landsat 8 C2 L2 SR.** TODAS las escenas de la provincia de Cusco, todo el año, desde 2013 hasta hoy. Se catalogan, se calculan sus métricas y se descargan, **cada una con su fecha** (en el nombre del archivo, en los metadatos del GeoTIFF y en un manifiesto).
- **Métricas e indicadores: MapBiomas Perú Colección 4**, de 2013 en adelante. TODOS los indicadores que MapBiomas entregue, en un CSV con la resolución temporal con la que vengan (anuales o mensuales). Los mapas de MapBiomas no se descargan.
- Descartadas: Landsat 5 y 7, Sentinel-2, GEOBOSQUES, WorldCover, SRTM y OpenStreetMap. Landsat 9 queda como opción (`INCLUIR_L9`).

**Cada cuánto toma Landsat 8.** Cada satélite vuelve a pasar sobre el mismo punto cada **16 días**. La "foto cada 8 días" se consigue **sumando Landsat 9** (en órbita desde 2021) y solo desde esa fecha, por eso `INCLUIR_L9 = False` por defecto. La sección 4 mide el intervalo real entre pasadas.

**Secciones**
1. Provincia de Cusco (de 01) y configuración.
2. Cobertura de años de cada fuente (de 03).
3. Qué entrega MapBiomas: documentación y descubrimiento de sus assets en GEE.
4. Catálogo de TODAS las escenas Landsat 8 desde 2013 (fecha, hora, nubosidad, % limpio, índices).
5. Indicadores de MapBiomas: cobertura (anual) 2013 → 2025 (de 04).
6. Indicadores de los demás productos de MapBiomas, anuales y mensuales, y CSV único.
7. Compuestos anuales jun–ago de Landsat 8 (insumo para LandTrendr) y sus métricas.
8. Tabla única por año (`metricas_por_anio.csv`) y guardado de todas las tablas.
9. Conclusiones y pendientes.
10. Descarga de escenas y compuestos, con su fecha (interruptores en `False` por defecto).

LandTrendr **no** se aplica aquí.

In [ ]:
from datetime import date
from pathlib import Path
import io
import math
import re

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

GEE_PROJECT = "viirs-peru"
MB_ASSET = "projects/mapbiomas-public/assets/peru/collection4/mapbiomas_peru_collection4_coverage_v1"  # bandas classification_YYYY
COLECCIONES = {"LC08": "LANDSAT/LC08/C02/T1_L2", "LC09": "LANDSAT/LC09/C02/T1_L2"}
INCLUIR_L9 = False   # True = suma Landsat 9 (desde 2021-11): una pasada cada 8 días en vez de cada 16

# Bandas (iguales en Landsat 8 y 9): nombre común -> banda original
BANDAS_L8 = {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"}
# Escala SR: reflectancia * 0.0000275 - 0.2
# Máscara QA_PIXEL: bits 0 a 4 (relleno, nube dilatada, cirrus, nube, sombra de nube)

ANIO_INI = 2013                                   # primer año de ambas fuentes
HOY = date.today()
ANIOS_L8 = list(range(ANIO_INI, HOY.year + 1))    # el año en curso es parcial
FECHA_INI, FECHA_FIN = "06-01", "09-01"           # estación seca para el compuesto: 1 jun -> 31 ago (fin exclusivo)
NUBES_MAX = 30                                    # PROPUESTA: CLOUD_COVER máximo para escena "útil"


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la raíz del proyecto (la que tiene el límite provincial).
    Se usa el GeoPackage como marcador porque puede existir una carpeta data/raw dentro de notebooks/."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg").is_file():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw/Provincia/PROVINCIA.gpkg desde esta ubicación.")


def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)


def rango_fechas(col):
    """Primera y última fecha de una colección, como texto."""
    d = ee.Dictionary({
        "min": col.aggregate_min("system:time_start"),
        "max": col.aggregate_max("system:time_start"),
    }).getInfo()
    if d["min"] is None:
        return "sin fecha"
    a = pd.to_datetime(d["min"], unit="ms").strftime("%Y-%m-%d")
    b = pd.to_datetime(d["max"], unit="ms").strftime("%Y-%m-%d")
    return f"{a} → {b}"


iniciar_gee()
RAIZ = buscar_raiz()
GPKG = RAIZ / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg"
gdf = gpd.read_file(GPKG)
prov = gdf[(gdf["ccdd"] == "08") & (gdf["ccpp"] == "01")]
assert len(prov) == 1, "El filtro de provincia no devolvió exactamente 1 registro"
aoi = ee.Geometry(prov.geometry.iloc[0].simplify(0.0001, preserve_topology=True).__geo_interface__)
mb = ee.Image(MB_ASSET)

col_base = ee.ImageCollection(COLECCIONES["LC08"])
if INCLUIR_L9:
    col_base = col_base.merge(ee.ImageCollection(COLECCIONES["LC09"]))
col_l8 = col_base.filterBounds(aoi).filterDate(f"{ANIO_INI}-01-01", f"{HOY.year + 1}-01-01")

print("Provincia:", prov["nombprov"].iloc[0], "| Raíz:", RAIZ)
print("Satélites:", "Landsat 8 + Landsat 9" if INCLUIR_L9 else "solo Landsat 8", "| Años Landsat:", ANIOS_L8[0], "–", ANIOS_L8[-1], f"(hoy: {HOY})")
CONFIG_LISTA = True   # marca que la configuración corrió; las demás secciones la comprueban

## 1 · Provincia de Cusco
Mismo filtro que en el notebook 01 (`ccdd = 08`, `ccpp = 01`). El área de referencia se calcula en UTM 19S (EPSG:32719).

In [ ]:
geom = prov.geometry.iloc[0]
area_utm_km2 = prov.to_crs(epsg=32719).area.iloc[0] / 1e6

print("Tipo de geometría:", geom.geom_type, "| Válida:", geom.is_valid, "| Vacía:", geom.is_empty)
print("Registros:", len(prov))
print(f"Área en EPSG:32719: {area_utm_km2:,.1f} km²")
print("Límites (lon/lat):", [round(v, 4) for v in prov.total_bounds])

In [ ]:
dpto = gdf[gdf["ccdd"] == "08"]
fig, ejes = plt.subplots(1, 2, figsize=(12, 6))
dpto.plot(ax=ejes[0], color="#e9e9e9", edgecolor="#8a8a8a", linewidth=0.5)
prov.plot(ax=ejes[0], color="#1a6fc0", edgecolor="white", linewidth=0.8)
ejes[0].set_title("Departamento de Cusco\n(provincia en azul)")
prov.plot(ax=ejes[1], color="#1a6fc0", edgecolor="white", linewidth=0.8)
ejes[1].set_title("Provincia de Cusco (ccdd=08, ccpp=01)")
for ax in ejes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## 2 · Cobertura de años por fuente

In [ ]:
# Guarda: esta celda necesita la configuración (primera celda de código).
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

print("Landsat sobre la provincia:", rango_fechas(col_l8))

filas = []
for anio in ANIOS_L8:
    n = col_l8.filterDate(f"{anio}-01-01", f"{anio + 1}-01-01").size().getInfo()
    filas.append({"año": anio, "escenas (año)": n})
df_l8 = pd.DataFrame(filas).set_index("año")
df_l8.T

In [ ]:
bandas_mb = mb.bandNames().getInfo()
anios_mb_todos = sorted(int(b.split("_")[1]) for b in bandas_mb if b.startswith("classification_"))
faltan_mb = [a for a in range(anios_mb_todos[0], anios_mb_todos[-1] + 1) if a not in anios_mb_todos]
anios_mb = [a for a in anios_mb_todos if a >= ANIO_INI]     # años que se analizan: 2013 -> último
proj_mb = mb.select(0).projection().getInfo()
escala_mb = mb.select(0).projection().nominalScale().getInfo()

print(f"MapBiomas C4 en el asset: {anios_mb_todos[0]} a {anios_mb_todos[-1]} ({len(anios_mb_todos)} años)")
print("Años intermedios faltantes:", faltan_mb or "ninguno")
print(f"Años que se analizan aquí: {anios_mb[0]} a {anios_mb[-1]} ({len(anios_mb)} años, {len(anios_mb) - 1} periodos)")
print("Resolución nominal (m):", round(escala_mb, 1))
print("Proyección nativa:", proj_mb.get("crs"), "| transform:", proj_mb.get("transform"))

In [ ]:
anios_todos = list(range(ANIO_INI, max(anios_mb[-1], ANIOS_L8[-1]) + 1))
matriz = pd.DataFrame(index=pd.Index(anios_todos, name="año"))
matriz["MapBiomas C4"] = [a in anios_mb for a in anios_todos]
matriz["Landsat (escenas)"] = [int(df_l8["escenas (año)"].get(a, 0)) for a in anios_todos]
matriz["año completo"] = [a < HOY.year for a in anios_todos]
matriz.replace({True: "✔", False: "—"})

## 3 · Qué entrega MapBiomas Perú (documentación y descubrimiento)
Fuente: [peru.mapbiomas.org](https://peru.mapbiomas.org/) (portada de la Colección 4, metodología, descargas y plataforma).

| Producto | Qué es | Uso en este notebook |
|---|---|---|
| **Cobertura y uso del suelo, Colección 4** | Mapas anuales de clases, 1985–2025, 30 m. La web indica **26 clases mapeadas** en el país, con dos clases nuevas de humedales (herbazal inundable de tierras bajas y altoandino). | Sección 5. Asset `classification_YYYY`, una banda por año. |
| Mosaicos Landsat | Mosaicos anuales de Landsat desde 1985, base de la clasificación. | No se usan: las imágenes son las escenas de Landsat 8. |
| Agua, Fuego | Iniciativas aparte con sus propios productos. | Sección 6, **solo si aparecen como assets en GEE**. |
| MapBiomas Alerta | Sistema de validación y refinamiento de alertas de deforestación. | No es una serie anual comparable. |
| Estadísticas | Tablas de la plataforma por provincia, distrito, etc. La web las describe como de la Colección 3. | No se usan: se recalculan con los assets de la Colección 4. |

**Cómo es el dato.** Cada año de la cobertura es **un solo mapa de clases** (sin día ni mes; no hay imágenes individuales que elegir). Se construye con los mosaicos anuales de Landsat. **La ventana de meses del mosaico y el clasificador están en el ATBD de la Colección 4 y no se verificaron: quedan por confirmar.** Datos bajo licencia CC BY 4.0: hay que citar a MapBiomas.

**Resolución temporal de los indicadores.** Algunos productos de MapBiomas son anuales y otros pueden ser mensuales. No se supone: las celdas siguientes **la detectan por los datos** (nombres de banda `..._YYYY` o `..._YYYY_MM`, o fechas de las imágenes de una colección) y dejan constancia en la tabla `productos_mapbiomas`.

In [ ]:
# Guarda: esta celda necesita la configuración (primera celda de código).
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

def listar_recursivo(carpeta, nivel=0, max_nivel=3):
    """Lista assets de GEE de forma recursiva (solo se entra en carpetas). Devuelve dicts con id, tipo y nivel."""
    salida, token = [], None
    while True:
        params = {"parent": carpeta}
        if token:
            params["pageToken"] = token
        try:
            resp = ee.data.listAssets(params)
        except Exception as e:
            print(f"  no pude listar {carpeta}: {repr(e)[:140]}")
            return salida
        for a in resp.get("assets", []):
            salida.append({"id": a["id"], "tipo": a["type"], "nivel": nivel})
            if a["type"] == "FOLDER" and nivel < max_nivel:
                salida += listar_recursivo(a["id"], nivel + 1, max_nivel)
        token = resp.get("nextPageToken")
        if not token:
            return salida


carpeta_peru = MB_ASSET.rsplit("/", 2)[0]       # projects/mapbiomas-public/assets/peru
assets_mb = listar_recursivo(carpeta_peru)
print(f"{carpeta_peru}: {len(assets_mb)} assets encontrados")
for a in assets_mb:
    print("  " * a["nivel"], f"{a['tipo']:17s}", a["id"])

In [ ]:
RE_MES = re.compile(r"((?:19|20)\d{2})[_-]?(0[1-9]|1[0-2])$")
RE_ANIO = re.compile(r"((?:19|20)\d{2})$")
MAX_ASSETS_DESCRIBIR = 80


def parsear_banda(nombre):
    """(año, mes) según el nombre de una banda; mes = None si es anual; None si no tiene fecha."""
    m = RE_MES.search(nombre)
    if m:
        return int(m.group(1)), int(m.group(2))
    a = RE_ANIO.search(nombre)
    if a:
        return int(a.group(1)), None
    return None


def describir_asset(id_, tipo):
    """Describe un asset de MapBiomas: bandas, resolución temporal, años desde 2013 y contenido. Devuelve (fila, info)."""
    fila = {"asset": id_, "tipo": tipo, "resolucion_temporal": "desconocida", "años_desde_2013": "",
            "bandas": "", "contenido": "", "calculable": "no", "motivo": ""}
    if tipo not in ("IMAGE", "IMAGE_COLLECTION"):
        fila["motivo"] = f"tipo {tipo}: no es imagen"
        return fila, None
    try:
        if tipo == "IMAGE_COLLECTION":
            col = ee.ImageCollection(id_)
            if col.size().getInfo() == 0:
                fila["motivo"] = "colección vacía"
                return fila, None
            primera = ee.Image(col.first())
            marcas = col.aggregate_array("system:time_start").getInfo()
        else:
            primera, marcas = ee.Image(id_), []
        bandas = primera.bandNames().getInfo()
        analisis = [parsear_banda(b) for b in bandas]
        con_fecha = [p for p in analisis if p]

        if con_fecha and len(con_fecha) >= 0.5 * len(bandas):
            res = "mensual (bandas)" if any(m for _, m in con_fecha) else "anual (bandas)"
            anios = sorted({a for a, _ in con_fecha if a >= ANIO_INI})
            muestra = next((b for b, p in zip(bandas, analisis) if p and p[0] >= ANIO_INI), bandas[0])
        elif marcas:
            fechas = pd.to_datetime(marcas, unit="ms")
            por_anio = pd.Series(fechas.year).value_counts()
            res = "mensual (colección)" if por_anio.median() > 1 else "anual (colección)"
            anios = sorted({int(y) for y in fechas.year if y >= ANIO_INI})
            muestra = bandas[0]
        else:
            res, anios, muestra = "desconocida", [], bandas[0]

        # Contenido: tipo de dato y, si es entero, histograma de una banda de muestra
        precision = primera.select(muestra).getInfo()["bands"][0]["data_type"]["precision"]
        esc = round(primera.select(muestra).projection().nominalScale().getInfo(), 1)
        if precision == "float":
            contenido = "continua"
        else:
            h = primera.select(muestra).reduceRegion(reducer=ee.Reducer.frequencyHistogram(), geometry=aoi,
                                                      scale=esc, maxPixels=1e13).getInfo()
            h = next(iter(h.values())) if h else None
            if not h:
                contenido = "sin datos en la provincia (banda de muestra)"
            else:
                valores = {float(k) for k in h}
                contenido = "binaria" if valores <= {0.0, 1.0} else ("categórica" if len(valores) <= 120 else "continua")
        fila.update({"resolucion_temporal": res, "años_desde_2013": f"{anios[0]}–{anios[-1]} ({len(anios)})" if anios else "ninguno",
                     "bandas": f"{len(bandas)}: {bandas[:3]}…", "contenido": contenido})
        ok = res.split(" ")[0] in ("anual", "mensual") and contenido in ("categórica", "binaria", "continua") and bool(anios)
        fila["calculable"] = "sí" if ok else "no"
        if not ok:
            fila["motivo"] = "sin resolución temporal detectable, sin años desde 2013 o sin datos en la provincia"
        info = {"id": id_, "tipo": tipo, "bandas": bandas, "parse": analisis, "res": res,
                "contenido": contenido, "esc": esc, "anios": anios}
        return fila, info
    except Exception as e:
        fila["motivo"] = f"error al describir: {repr(e)[:140]}"
        return fila, None


candidatos = [a for a in assets_mb if a["tipo"] in ("IMAGE", "IMAGE_COLLECTION", "TABLE")]
if len(candidatos) > MAX_ASSETS_DESCRIBIR:
    print(f"⚠ {len(candidatos)} assets: se describen solo los primeros {MAX_ASSETS_DESCRIBIR}. Sube MAX_ASSETS_DESCRIBIR si hace falta.")
    candidatos = candidatos[:MAX_ASSETS_DESCRIBIR]

filas_prod, infos_assets = [], []
for a in candidatos:
    fila, info = describir_asset(a["id"], a["tipo"])
    filas_prod.append(fila)
    if info:
        infos_assets.append(info)

productos_mapbiomas = pd.DataFrame(filas_prod)
print("Productos descritos:", len(productos_mapbiomas), "| calculables:", int((productos_mapbiomas["calculable"] == "sí").sum()) if len(productos_mapbiomas) else 0)
productos_mapbiomas

In [ ]:
print("Bandas del asset de cobertura:", len(bandas_mb), "| ejemplo:", bandas_mb[:2], "...", bandas_mb[-1])
print("Propiedades del asset de cobertura:", ee.Image(MB_ASSET).propertyNames().getInfo())
if len(productos_mapbiomas) and (productos_mapbiomas["calculable"] == "sí").sum() <= 1:
    print("Aparte de la cobertura no hay otros productos calculables en GEE. Los demás (fuego, agua, etc.), si existen, "
          "solo están publicados en la plataforma de MapBiomas Perú (https://plataforma.mapbiomas.org/projects/mapbiomas/peru): "
          "habrá que bajarlos a mano y se documentan como pendiente.")

## 4 · Catálogo de TODAS las escenas Landsat 8, 2013 → hoy
Todas las escenas que tocan la provincia, todo el año. Por escena: **fecha y hora de la toma (UTC)**, path/row, nubosidad, **% de la provincia con píxel limpio** (QA_PIXEL bits 0–4) y **media de NDVI, EVI, NBR y NDMI** sobre los píxeles limpios.

Solo se cataloga: aquí no se descarga nada. Es la parte más lenta del notebook (varias consultas a GEE por año).

In [ ]:
# Guarda: esta celda necesita la configuración (primera celda de código).
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

INDICES = ["NDVI", "EVI", "NBR", "NDMI"]
PROPS = ["DATE_ACQUIRED", "SCENE_CENTER_TIME", "WRS_PATH", "WRS_ROW", "CLOUD_COVER", "CLOUD_COVER_LAND"]


def escalar_y_enmascarar(img):
    """Reflectancia escalada, con los nombres comunes de banda y la máscara QA_PIXEL (bits 0–4)."""
    limpio = img.select("QA_PIXEL").bitwiseAnd(31).eq(0)
    sr = (img.select(list(BANDAS_L8.values()), list(BANDAS_L8.keys()))
          .multiply(0.0000275).add(-0.2))
    return sr.updateMask(limpio).copyProperties(img, ["system:time_start"])


def agregar_indices(sr):
    """Añade NDVI, EVI, NBR y NDMI a una imagen con bandas blue, red, nir, swir1 y swir2."""
    ndvi = sr.normalizedDifference(["nir", "red"]).rename("NDVI")
    nbr = sr.normalizedDifference(["nir", "swir2"]).rename("NBR")
    ndmi = sr.normalizedDifference(["nir", "swir1"]).rename("NDMI")
    evi = sr.expression("2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)",
                        {"NIR": sr.select("nir"), "RED": sr.select("red"), "BLUE": sr.select("blue")}).rename("EVI")
    return sr.addBands([ndvi, evi, nbr, ndmi])


def pct_limpio(img):
    """Fracción de la provincia con píxel limpio. Lo que la escena no cubre cuenta como no limpio."""
    limpio = img.select("QA_PIXEL").bitwiseAnd(31).eq(0).unmask(0).rename("limpio")
    return limpio.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e13).get("limpio")


def ficha_escena(img):
    """Propiedades, % limpio y media de índices de una escena, como Feature."""
    img = ee.Image(img)
    idx = agregar_indices(escalar_y_enmascarar(img)).select(INDICES)
    medias = idx.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e13)
    base = (img.toDictionary(PROPS)
            .set("indice", img.get("system:index"))
            .set("inicio_utc", img.get("system:time_start"))
            .set("limpio", pct_limpio(img)))
    return ee.Feature(None, base.combine(medias))


def catalogo_anio(anio):
    col = col_l8.filterDate(f"{anio}-01-01", f"{anio + 1}-01-01")
    fc = ee.FeatureCollection(col.toList(col.size()).map(ficha_escena))
    return [f["properties"] for f in fc.getInfo()["features"]]


filas = []
for anio in ANIOS_L8:
    for p in catalogo_anio(anio):
        filas.append({"año": anio, **p})
    print(anio, "→", sum(1 for f in filas if f["año"] == anio), "escenas")

escenas = pd.DataFrame(filas).rename(columns={
    "indice": "escena", "DATE_ACQUIRED": "fecha", "SCENE_CENTER_TIME": "hora_centro", "WRS_PATH": "path", "WRS_ROW": "row",
    "CLOUD_COVER": "nubes_%", "CLOUD_COVER_LAND": "nubes_tierra_%"})
escenas["limpio_%"] = (escenas["limpio"] * 100).round(1)
escenas = escenas.drop(columns="limpio").rename(columns={i: f"{i}_media" for i in INDICES})
escenas["fecha"] = pd.to_datetime(escenas["fecha"])
escenas["inicio_utc"] = pd.to_datetime(escenas["inicio_utc"], unit="ms", utc=True)
escenas["mes"] = escenas["fecha"].dt.month
escenas["en_jun_ago"] = escenas["mes"].between(6, 8)
escenas["util"] = escenas["nubes_%"] < NUBES_MAX
escenas["satelite"] = escenas["escena"].str[:4]
escenas = escenas.sort_values(["fecha", "path", "row"]).reset_index(drop=True)
print("Escenas en total:", len(escenas), "| útiles (nubes <", NUBES_MAX, "%):", int(escenas["util"].sum()))
escenas.head(10)

### 4.1 · ¿Cada cuánto pasa el satélite?
Intervalo entre fechas consecutivas de una misma escena (path/row). El teórico de Landsat 8 es de 16 días; si falta alguna pasada en el catálogo, el intervalo sale de 32 o más.

In [ ]:
def intervalos(df):
    d = df.sort_values("fecha").groupby(["satelite", "path", "row"])["fecha"].diff().dt.days.dropna()
    return d.astype(int)


gaps = intervalos(escenas)
print("Escenas distintas (path/row):", escenas.groupby(["path", "row"]).ngroups)
print("Intervalo más frecuente (días):", int(gaps.mode().iloc[0]), "| mediana:", float(gaps.median()))
print("Distribución de intervalos (días: n.º de veces):")
print(gaps.value_counts().head(6).to_string())
print()
print("Fechas distintas con alguna escena:", escenas["fecha"].nunique(), "| fechas por año (media):",
      round(escenas.groupby("año")["fecha"].nunique().mean(), 1))

### 4.2 · Resumen por año y por mes

In [ ]:
resumen = escenas.groupby("año").agg(
    escenas=("escena", "count"),
    utiles=("util", "sum"),
    nubes_min=("nubes_%", "min"),
    nubes_mediana=("nubes_%", "median"),
    nubes_max=("nubes_%", "max"),
    limpio_medio=("limpio_%", "mean"),
    en_jun_ago=("en_jun_ago", "sum"),
).reindex(ANIOS_L8).fillna(0)
resumen["limpio_medio"] = resumen["limpio_medio"].round(1)
resumen["año parcial"] = np.where(resumen.index >= HOY.year, "parcial", "")
resumen

In [ ]:
por_mes = escenas.pivot_table(index="año", columns="mes", values="escena", aggfunc="count", fill_value=0).reindex(ANIOS_L8, fill_value=0)
por_mes

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
ejes[0].bar(resumen.index, resumen["escenas"], color="#b9c6d6", label="todas")
ejes[0].bar(resumen.index, resumen["utiles"], color="#1a6fc0", label=f"CLOUD_COVER < {NUBES_MAX}")
ejes[0].set_title("Escenas por año")
ejes[0].legend()
ejes[1].plot(por_mes.columns, por_mes.mean(), marker="o", color="#1a6fc0")
ejes[1].set_title("Escenas por mes (media de años)")
ejes[1].set_xticks(range(1, 13))
ejes[2].plot(resumen.index, resumen["limpio_medio"], marker="o", color="#1a6fc0")
ejes[2].set_title("% de la provincia limpio (media de escenas)")
ejes[2].set_ylim(0, 100)
for ax in (ejes[0], ejes[2]):
    ax.set_xticks(resumen.index)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 5 · Indicadores de MapBiomas, 2013 → 2025
Áreas con `pixelArea`, sin `bestEffort`. Los nombres de clase salen de mi notebook 04; las clases que no estén ahí quedan "(sin nombre verificado)" hasta contrastarlas con la leyenda oficial de MapBiomas Perú C4.

In [ ]:
# Guarda: esta celda necesita la configuración (primera celda de código).
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

# Nombres de mi notebook 04. Las clases que no estén aquí quedan marcadas como sin verificar.
NOMBRE_MB = {
    3: "Bosque", 4: "Bosque seco", 5: "Manglar", 6: "Bosque inundable",
    9: "Plantación forestal", 12: "Formación herbácea", 15: "Pasto (beta)",
    21: "Mosaico agropecuario", 24: "Infraestructura urbana", 31: "Acuicultura",
    33: "Río, lago u océano", 66: "Matorral y otros arbustales",
    68: "Otra área natural sin vegetación", 82: "Herbazal inundable altoandino",
}

# PROPUESTA de mi notebook 04. Se revisa en la sección 5.2 con los datos reales.
GRUPOS = {
    "bosque": [3, 4, 5, 6],
    "agricola": [21],
    "pastos": [12, 15],
    "matorral": [66],
    "urbano": [24],
    "agua": [31, 33],
}


def grupo_de(codigo):
    for grupo, codigos in GRUPOS.items():
        if codigo in codigos:
            return grupo
    return "otras"


def nombre_clase(codigo):
    return NOMBRE_MB.get(codigo, "(sin nombre verificado)")


def area_por_clase(anio):
    """Área (km²) por clase de MapBiomas en la provincia, para un año."""
    clase = mb.select(f"classification_{anio}")
    grupos = (ee.Image.pixelArea().addBands(clase)
              .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="codigo"),
                            geometry=aoi, scale=30, maxPixels=1e13)
              .get("groups").getInfo())
    return {int(g["codigo"]): g["sum"] / 1e6 for g in grupos}


areas = {anio: area_por_clase(anio) for anio in anios_mb}
tabla_area = pd.DataFrame(areas).T.fillna(0.0).sort_index(axis=1)
tabla_area.index.name = "año"
print(f"Años: {tabla_area.index.min()}–{tabla_area.index.max()} | Clases presentes en la provincia: {len(tabla_area.columns)} -> {list(tabla_area.columns)}")

### 5.1 · Área por clase y año

In [ ]:
tabla_rotulada = tabla_area.copy()
tabla_rotulada.columns = [f"{c} {nombre_clase(c)}" for c in tabla_area.columns]
tabla_rotulada.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.stackplot(tabla_area.index, tabla_area.T.values, labels=tabla_rotulada.columns)
ax.set_title("Área por clase de MapBiomas en la provincia de Cusco (km²)")
ax.set_xlabel("año")
ax.set_ylabel("km²")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
plt.tight_layout()
plt.show()

siempre_vacias = [c for c in tabla_area.columns if tabla_area[c].max() == 0]
print("Clases siempre en cero:", siempre_vacias or "ninguna")

### 5.2 · ¿Qué es "bosque" en Cusco?
Mis docs dicen que el código 3 tiene solo 1.8 km² en la provincia. Aquí se comprueba con los datos y se revisa la agrupación propuesta.

In [ ]:
ultimo = anios_mb[-1]
top = tabla_area.loc[ultimo].sort_values(ascending=False)
detalle = pd.DataFrame({
    "clase": [f"{c} {nombre_clase(c)}" for c in top.index],
    f"km² {ultimo}": top.values.round(2),
    "grupo propuesto": [grupo_de(c) for c in top.index],
})
display(detalle)

cols_bosque = [c for c in tabla_area.columns if grupo_de(c) == "bosque"]
bosque_km2 = tabla_area[cols_bosque].sum(axis=1)
print(f"Grupo 'bosque' {GRUPOS['bosque']}: {bosque_km2.loc[ultimo]:.1f} km² en {ultimo} "
      f"({100 * bosque_km2.loc[ultimo] / area_utm_km2:.1f} % de la provincia)")
print("Código 3 solo:", round(float(tabla_area[3].loc[ultimo]), 2) if 3 in tabla_area.columns else 0, "km²")
if bosque_km2.loc[ultimo] < 0.05 * area_utm_km2:
    print("⚠ El grupo 'bosque' es menor al 5 % de la provincia: revisar GRUPOS antes de interpretar la pérdida de bosque.")

### 5.3 · Transiciones por periodo (matriz completa)
Para cada par de años consecutivos se guarda la tabla larga clase_origen → clase_destino con su área en km² (`pixelArea`, para que cuadre con la tabla de áreas).

In [ ]:
def transicion_larga(anio_a, anio_b):
    """Tabla larga de transiciones entre dos años de MapBiomas (km²)."""
    a = mb.select(f"classification_{anio_a}").toInt32()
    b = mb.select(f"classification_{anio_b}").toInt32()
    par = a.multiply(100).add(b).rename("par")
    grupos = (ee.Image.pixelArea().addBands(par)
              .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="par"),
                            geometry=aoi, scale=30, maxPixels=1e13)
              .get("groups").getInfo())
    filas = []
    for g in grupos:
        c1, c2 = divmod(int(g["par"]), 100)
        filas.append({"anio_a": anio_a, "anio_b": anio_b, "clase_origen": c1,
                      "clase_destino": c2, "km2": g["sum"] / 1e6})
    return filas


filas = []
for anio in anios_mb[:-1]:
    filas += transicion_larga(anio, anio + 1)
trans = pd.DataFrame(filas)
trans["periodo"] = trans["anio_a"].astype(str) + " → " + trans["anio_b"].astype(str)
trans["grupo_origen"] = trans["clase_origen"].map(grupo_de)
trans["grupo_destino"] = trans["clase_destino"].map(grupo_de)
print("Filas de la tabla larga:", len(trans), "| Periodos:", trans["periodo"].nunique())
trans.head()

### 5.4 · Indicadores por periodo
- **Pérdida de bosque:** bosque en t → no bosque en t+1.
- **Expansión agrícola:** no agrícola en t → agrícola en t+1.
- **Expansión de bosque:** no bosque en t → bosque en t+1.
- **Balance neto de bosque:** expansión − pérdida.
- **Área que cambió de clase** y **área estable** (intensidad de cambio), en km² y %.
- **Tasa de pérdida (%/año):** pérdida del periodo entre el bosque al inicio del periodo.

In [ ]:
def km2(mask):
    return trans[mask].groupby("anio_a")["km2"].sum()


ind = pd.DataFrame(index=pd.Index(anios_mb[:-1], name="año_inicio"))
ind["periodo"] = [f"{a} → {a + 1}" for a in ind.index]
ind["pérdida de bosque (km²)"] = km2((trans.grupo_origen == "bosque") & (trans.grupo_destino != "bosque"))
ind["expansión agrícola (km²)"] = km2((trans.grupo_origen != "agricola") & (trans.grupo_destino == "agricola"))
ind["expansión de bosque (km²)"] = km2((trans.grupo_origen != "bosque") & (trans.grupo_destino == "bosque"))
ind["área que cambió de clase (km²)"] = km2(trans.clase_origen != trans.clase_destino)
ind["área estable (km²)"] = km2(trans.clase_origen == trans.clase_destino)
ind = ind.fillna(0.0)
ind["balance neto de bosque (km²)"] = ind["expansión de bosque (km²)"] - ind["pérdida de bosque (km²)"]

for c in ["pérdida de bosque", "expansión agrícola", "expansión de bosque", "área que cambió de clase"]:
    ind[f"{c} (% provincia)"] = 100 * ind[f"{c} (km²)"] / area_utm_km2

bosque_inicio = bosque_km2.reindex(ind.index)
ind["bosque al inicio (km²)"] = bosque_inicio
ind["tasa de pérdida (%/año)"] = np.where(bosque_inicio > 0, 100 * ind["pérdida de bosque (km²)"] / bosque_inicio, np.nan)
ind["pérdida acumulada (km²)"] = ind["pérdida de bosque (km²)"].cumsum()
ind.round(3)

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
ejes[0].bar(ind.index, ind["pérdida de bosque (km²)"], color="#b5402a")
ejes[0].set_title("Pérdida de bosque (km²)")
ejes[1].bar(ind.index, ind["expansión agrícola (km²)"], color="#c28a1d")
ejes[1].set_title("Expansión agrícola (km²)")
ejes[2].plot(ind.index, ind["pérdida acumulada (km²)"], color="#b5402a", marker="o")
ejes[2].set_title("Pérdida de bosque acumulada (km²)")
for ax in ejes:
    ax.set_xlabel("año de inicio del periodo")
plt.tight_layout()
plt.show()

### 5.5 · Matriz acumulada
Primer año → último año, calculada directamente entre los dos mapas (no sumando periodos), para ver el cambio neto.

In [ ]:
acum = pd.DataFrame(transicion_larga(anios_mb[0], anios_mb[-1]))
matriz_acum = acum.pivot_table(index="clase_origen", columns="clase_destino", values="km2", aggfunc="sum", fill_value=0.0)
matriz_acum.index = [f"{c} {nombre_clase(c)}" for c in matriz_acum.index]
matriz_acum.columns = [f"{c} {nombre_clase(c)}" for c in matriz_acum.columns]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(np.log10(matriz_acum.values + 1e-3), cmap="viridis")
ax.set_xticks(range(len(matriz_acum.columns)), matriz_acum.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(matriz_acum.index)), matriz_acum.index, fontsize=7)
ax.set_title(f"Transiciones {anios_mb[0]} → {anios_mb[-1]} (log10 km²)")
ax.set_xlabel("clase destino")
ax.set_ylabel("clase origen")
fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()
matriz_acum.round(2)

### 5.6 · Chequeos de consistencia

In [ ]:
area_gee_km2 = aoi.area().divide(1e6).getInfo()
suma_clases = tabla_area.sum(axis=1)
suma_trans = trans.groupby("anio_a")["km2"].sum()

print(f"Área provincia UTM 19S:               {area_utm_km2:8.2f} km²")
print(f"Área del polígono simplificado (GEE): {area_gee_km2:8.2f} km²  (dif. {area_gee_km2 - area_utm_km2:+.2f})")
print(f"Suma de clases por año (pixelArea):   mín {suma_clases.min():.2f} | máx {suma_clases.max():.2f} km²")
print(f"Suma de la matriz por periodo:        mín {suma_trans.min():.2f} | máx {suma_trans.max():.2f} km²")
print("¿Suma de clases constante entre años?", bool(np.allclose(suma_clases, suma_clases.iloc[0], rtol=1e-3)))
print("¿Suma de la matriz = suma de clases del año inicial?",
      bool(np.allclose(suma_trans.values, suma_clases.reindex(suma_trans.index).values, rtol=1e-3)))
print("NaN en la tabla de áreas:", int(tabla_area.isna().sum().sum()),
      "| periodos sin bosque (tasa NaN):", int(ind["tasa de pérdida (%/año)"].isna().sum()))

### 5.7 · Cobertura en formato largo
Todos los indicadores de cobertura (áreas por clase y año, e indicadores de cambio por periodo) se pasan a formato largo para unirlos con los demás productos en la sección 6. El indicador de un periodo t−1 → t se asigna al año t y el periodo queda en `observacion`.

In [ ]:
def partir_unidad(nombre):
    """Separa 'pérdida de bosque (km²)' en ('pérdida de bosque', 'km²')."""
    m = re.match(r"^(.*) \((.*)\)$", nombre)
    return (m.group(1), m.group(2)) if m else (nombre, "")


filas = []
for anio, fila in tabla_area.iterrows():
    for c, v in fila.items():
        filas.append({"producto": "cobertura", "asset": MB_ASSET, "indicador": "área por clase", "resolucion_temporal": "anual",
                      "año": anio, "mes": None, "clase_codigo": c, "clase_nombre": nombre_clase(c),
                      "valor": v, "unidad": "km²", "observacion": ""})

cambio = ind.reset_index().melt(id_vars=["año_inicio", "periodo"], var_name="col", value_name="valor")
for r in cambio.itertuples():
    nombre, unidad = partir_unidad(r.col)
    filas.append({"producto": "cobertura", "asset": MB_ASSET, "indicador": nombre, "resolucion_temporal": "anual",
                  "año": r.año_inicio + 1, "mes": None, "clase_codigo": None, "clase_nombre": "",
                  "valor": r.valor, "unidad": unidad, "observacion": f"periodo {r.periodo}"})

largo_cobertura = pd.DataFrame(filas)
print("Indicadores de cobertura en formato largo:", len(largo_cobertura), "filas")
largo_cobertura.head()

## 6 · Indicadores de los demás productos de MapBiomas (anuales y mensuales)
Para cada producto marcado como **calculable** en la sección 3 se calculan sus indicadores sobre la provincia desde 2013, **respetando su resolución temporal**: un valor por año si es anual y uno por mes si es mensual.

- **Categórico:** área (km²) por clase.
- **Binario** (por ejemplo, área afectada): área (km²) con valor 1.
- **Continuo:** media sobre la provincia, con la unidad **sin verificar** (queda anotado).
- Si un producto no se sabe interpretar, queda como "no calculado" y **no se le inventa ningún indicador**.

Todo se junta con la cobertura en un único CSV en formato largo: `mapbiomas_indicadores_todos.csv`.

In [ ]:
# Guarda: esta celda necesita la configuración (primera celda de código).
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

def filas_valor(img, contenido, esc, base):
    """Indicadores de una imagen de una banda sobre la provincia. `base` lleva producto, asset, resolución, año y mes."""
    if contenido == "categórica":
        grupos = (ee.Image.pixelArea().addBands(img.rename("v"))
                  .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="codigo"),
                                geometry=aoi, scale=esc, maxPixels=1e13)
                  .get("groups").getInfo())
        return [{**base, "indicador": "área por clase", "clase_codigo": int(g["codigo"]),
                 "clase_nombre": "(sin nombre verificado)", "valor": g["sum"] / 1e6, "unidad": "km²"} for g in grupos]
    if contenido == "binaria":
        r = (ee.Image.pixelArea().updateMask(img.eq(1))
             .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi, scale=esc, maxPixels=1e13).get("area").getInfo())
        return [{**base, "indicador": "área con valor 1", "clase_codigo": 1, "clase_nombre": "",
                 "valor": (r or 0) / 1e6, "unidad": "km²"}]
    d = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=esc, maxPixels=1e13).getInfo()
    valor = next(iter(d.values())) if d else None
    return [{**base, "indicador": "media sobre la provincia", "clase_codigo": None, "clase_nombre": "",
             "valor": valor, "unidad": "sin verificar", "observacion": (base.get("observacion", "") + " | valor continuo: unidad no verificada").strip(" |")}]


def calcular_producto(info):
    """Todas las filas de indicadores de un producto, desde ANIO_INI, con su resolución temporal."""
    producto = info["id"].rstrip("/").split("/")[-1]
    filas = []
    if info["res"].endswith("(bandas)"):
        img = ee.Image(info["id"])
        for b, p in zip(info["bandas"], info["parse"]):
            if not p or p[0] < ANIO_INI:
                continue
            base = {"producto": producto, "asset": info["id"], "resolucion_temporal": "mensual" if p[1] else "anual",
                    "año": p[0], "mes": p[1], "observacion": f"banda {b}"}
            filas += filas_valor(img.select(b), info["contenido"], info["esc"], base)
    else:
        col = ee.ImageCollection(info["id"]).filterDate(f"{ANIO_INI}-01-01", f"{HOY.year + 1}-01-01")
        n = col.size().getInfo()
        lista = col.toList(n)
        mensual = info["res"].startswith("mensual")
        for i in range(n):
            im = ee.Image(lista.get(i))
            t = pd.to_datetime(im.get("system:time_start").getInfo(), unit="ms")
            base = {"producto": producto, "asset": info["id"], "resolucion_temporal": "mensual" if mensual else "anual",
                    "año": t.year, "mes": t.month if mensual else None,
                    "observacion": f"imagen {im.get('system:index').getInfo()}"}
            filas += filas_valor(im.select(0), info["contenido"], info["esc"], base)
    return filas


a_calcular = [i for i in infos_assets if i["id"] != MB_ASSET and i["res"].split(" ")[0] in ("anual", "mensual")
              and i["contenido"] in ("categórica", "binaria", "continua") and i["anios"]]
print("Productos a calcular (sin la cobertura, ya hecha en la sección 5):", [i["id"].split("/")[-1] for i in a_calcular] or "ninguno")

filas_otros, fallos = [], []
for info in a_calcular:
    try:
        nuevas = calcular_producto(info)
        filas_otros += nuevas
        print(f"  {info['id'].split('/')[-1]}: {len(nuevas)} filas ({info['res']})")
    except Exception as e:
        fallos.append((info["id"], repr(e)[:160]))
        print(f"  ✗ {info['id'].split('/')[-1]}: {repr(e)[:160]}")

largo_otros = pd.DataFrame(filas_otros)
no_calculados = [(r["asset"], r["motivo"] or r["contenido"]) for _, r in productos_mapbiomas.iterrows()
                 if r["calculable"] != "sí" and r["asset"] != MB_ASSET]

In [ ]:
mapbiomas_largo = pd.concat([largo_cobertura, largo_otros], ignore_index=True)
columnas = ["producto", "asset", "indicador", "resolucion_temporal", "año", "mes", "clase_codigo", "clase_nombre", "valor", "unidad", "observacion"]
mapbiomas_largo = mapbiomas_largo.reindex(columns=columnas)
mapbiomas_largo["año"] = mapbiomas_largo["año"].astype("Int64")
mapbiomas_largo["mes"] = mapbiomas_largo["mes"].astype("Int64")

print("Filas en total:", len(mapbiomas_largo))
print("Productos:", mapbiomas_largo["producto"].nunique(), "| indicadores distintos:", mapbiomas_largo[["producto", "indicador"]].drop_duplicates().shape[0])
print("Por resolución temporal:")
print(mapbiomas_largo.groupby("resolucion_temporal")[["producto", "indicador"]].nunique().to_string())
print("No calculados:", no_calculados or "ninguno")
print("Fallos:", fallos or "ninguno")
mapbiomas_largo.head()

## 7 · Compuestos anuales de Landsat 8, jun–ago
Aunque se catalogan y descargan todas las escenas, LandTrendr necesitará **un valor por píxel y por año**. Para eso se define un compuesto por año: **mediana** de los píxeles limpios de las escenas de **jun–ago** (estación seca, menos nubes) con `CLOUD_COVER < NUBES_MAX`.

Bandas: `blue, green, red, nir, swir1, swir2` (reflectancia), `NDVI, EVI, NBR, NDMI` y `n_obs` (observaciones limpias por píxel). Los índices van como **bandas**, no como métricas: la elección del que usará LandTrendr se hace después.

In [ ]:
# Guarda: esta celda necesita la configuración (primera celda de código).
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

def compuesto_anual(anio):
    """Compuesto jun–ago de un año: mediana + índices + n_obs, recortado a la provincia."""
    col = (col_l8.filterDate(f"{anio}-{FECHA_INI}", f"{anio}-{FECHA_FIN}")
           .filter(ee.Filter.lt("CLOUD_COVER", NUBES_MAX))
           .map(escalar_y_enmascarar))
    n_obs = col.count().select("blue").rename("n_obs")
    return (agregar_indices(col.median()).addBands(n_obs)
            .toFloat().clip(aoi).set("anio", anio))


# El compuesto de un año solo se arma cuando jun–ago ya terminó
utiles_jja = escenas[escenas["en_jun_ago"] & escenas["util"]].groupby("año").size().to_dict()
anios_con_compuesto = [a for a in ANIOS_L8 if utiles_jja.get(a, 0) > 0 and (a < HOY.year or HOY >= date(a, 9, 1))]
sin_compuesto = [a for a in ANIOS_L8 if a not in anios_con_compuesto]
print("Escenas útiles jun–ago por año:", {a: utiles_jja.get(a, 0) for a in ANIOS_L8})
print("Años con compuesto:", anios_con_compuesto)
print("Años SIN compuesto (sin escenas útiles o jun–ago sin terminar):", sin_compuesto or "ninguno")
print("Bandas:", compuesto_anual(anios_con_compuesto[0]).bandNames().getInfo())

In [ ]:
def miniatura(img, vis, ancho=500):
    """Imagen PNG de una miniatura de GEE, como arreglo para matplotlib."""
    url = img.visualize(**vis).getThumbURL({"region": aoi, "dimensions": ancho, "format": "png", "crs": "EPSG:32719"})
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    return plt.imread(io.BytesIO(r.content), format="png")


vis_rgb = {"bands": ["red", "green", "blue"], "min": 0.0, "max": 0.3}
vis_ndvi = {"bands": ["NDVI"], "min": 0.0, "max": 0.9, "palette": ["#a6611a", "#dfc27d", "#f5f5f5", "#80cdc1", "#018571"]}
muestra = [a for a in (2013, 2019, anios_con_compuesto[-1]) if a in anios_con_compuesto]

fig, ejes = plt.subplots(2, len(muestra), figsize=(5 * len(muestra), 9), squeeze=False)
for j, anio in enumerate(muestra):
    img = compuesto_anual(anio)
    ejes[0][j].imshow(miniatura(img, vis_rgb))
    ejes[0][j].set_title(f"{anio} · RGB")
    ejes[1][j].imshow(miniatura(img, vis_ndvi))
    ejes[1][j].set_title(f"{anio} · NDVI")
for ax in ejes.ravel():
    ax.set_axis_off()
plt.tight_layout()
plt.show()
print("Revisar a ojo: que no haya saltos fuertes de color entre años que parezcan cambio de cobertura.")

### 7.1 · Métricas de cada compuesto anual
Media, mediana y desviación estándar de cada banda e índice sobre los píxeles de la provincia, más el % de la provincia con al menos una observación limpia.

In [ ]:
REDUCTOR = (ee.Reducer.mean()
            .combine(ee.Reducer.median(), sharedInputs=True)
            .combine(ee.Reducer.stdDev(), sharedInputs=True))


def metricas_compuesto(anio):
    """Media, mediana y desviación de cada banda del compuesto, y % de provincia con observaciones."""
    img = compuesto_anual(anio)
    est = img.reduceRegion(reducer=REDUCTOR, geometry=aoi, scale=30, maxPixels=1e13)
    cobertura = (img.select("n_obs").gt(0).unmask(0).rename("cobertura")
                 .reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e13))
    d = ee.Dictionary(est).combine(cobertura).getInfo()
    cob = 100 * d.pop("cobertura")
    out = {f"L8 compuesto {k.replace('_', ' ')}": v for k, v in d.items()}
    out["L8 compuesto % provincia con datos"] = cob
    return {"año": anio, **out}


met_compuestos = pd.DataFrame([metricas_compuesto(a) for a in anios_con_compuesto]).set_index("año")
print("Métricas calculadas:", met_compuestos.shape[1], "columnas,", len(met_compuestos), "años")
met_compuestos.filter(like="NDVI").round(3)

## 8 · Tabla única de métricas por año y guardado
Una fila por año (2013 → hoy) con **todas** las métricas de ese año:
- **MapBiomas:** áreas por clase y por grupo en el año t, y los indicadores del periodo que **termina** en t (t−1 → t).
- **Landsat 8:** resumen de TODAS las escenas del año, de las útiles, y de las de jun–ago útiles (escenas, nubosidad, % limpio y media de índices), más las métricas del compuesto jun–ago.

Los indicadores **mensuales** (si los hay) no caben en esta tabla anual: van en `mapbiomas_indicadores_todos.csv`, que conserva la resolución temporal de cada producto.

Lo que una fuente no tiene queda vacío: MapBiomas no tiene años posteriores a 2025, y 2013 solo tiene indicadores de periodo en Landsat y áreas en MapBiomas (el primer periodo termina en 2014).

In [ ]:
# Guarda: esta celda necesita la configuración (primera celda de código).
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

# MapBiomas: áreas por clase y por grupo, en el año t
mb_areas = tabla_area.copy()
mb_areas.columns = [f"MB área {c} {nombre_clase(c)} (km²)" for c in mb_areas.columns]
mb_grupos = pd.DataFrame({f"MB área grupo {g} (km²)": tabla_area[[c for c in tabla_area.columns if grupo_de(c) == g]].sum(axis=1)
                          for g in list(GRUPOS) + ["otras"]})

# MapBiomas: indicadores del periodo que termina en t (t-1 → t)
mb_cambio = ind.drop(columns="periodo").copy()
mb_cambio.index = mb_cambio.index + 1
mb_cambio.index.name = "año"
mb_cambio.columns = [f"MB {c} (t-1→t)" for c in mb_cambio.columns]


def resumen_escenas(df, etiqueta):
    """Escenas, nubosidad, % limpio y media de índices por año, para un subconjunto de escenas."""
    g = df.groupby("año")
    out = pd.DataFrame({
        f"L8 escenas {etiqueta}": g["escena"].count(),
        f"L8 nubes mediana {etiqueta} (%)": g["nubes_%"].median(),
        f"L8 % limpio medio {etiqueta}": g["limpio_%"].mean(),
    })
    for i in INDICES:
        out[f"L8 {i} media {etiqueta}"] = g[f"{i}_media"].mean()
    return out


l8_todas = resumen_escenas(escenas, "(todas)")
l8_utiles = resumen_escenas(escenas[escenas["util"]], f"(nubes<{NUBES_MAX})")
l8_jja = resumen_escenas(escenas[escenas["util"] & escenas["en_jun_ago"]], f"(jun–ago, nubes<{NUBES_MAX})")

anios_tabla = list(range(ANIO_INI, max(anios_mb[-1], ANIOS_L8[-1]) + 1))
metricas_anio = (pd.concat([mb_areas, mb_grupos, mb_cambio, l8_todas, l8_utiles, l8_jja, met_compuestos], axis=1)
                 .reindex(anios_tabla))
metricas_anio.index.name = "año"
metricas_anio["L8 año parcial"] = [a >= HOY.year for a in metricas_anio.index]
print("Tabla única por año:", metricas_anio.shape[0], "filas ×", metricas_anio.shape[1], "columnas")
vista = [c for c in metricas_anio.columns if c.startswith(("MB área grupo bosque", "MB pérdida de bosque (km²)", "L8 escenas", "L8 NDVI media (todas)", "L8 compuesto NDVI mean"))]
metricas_anio[vista].round(3)

In [ ]:
SALIDA = RAIZ / "data" / "processed"
SALIDA.mkdir(parents=True, exist_ok=True)

archivos = {
    "metricas_por_anio.csv": metricas_anio.reset_index(),
    "mapbiomas_indicadores_todos.csv": mapbiomas_largo,
    "productos_mapbiomas.csv": productos_mapbiomas,
    "mapbiomas_areas_clase_anio.csv": tabla_rotulada.reset_index(),
    "mapbiomas_indicadores_periodo.csv": ind.reset_index(),
    "mapbiomas_transiciones_largo.csv": trans,
    "mapbiomas_matriz_acumulada.csv": matriz_acum.reset_index(names="clase_origen"),
    "landsat8_escenas.csv": escenas,
    "landsat8_metricas_compuestos.csv": met_compuestos.reset_index(),
}
for nombre, df in archivos.items():
    df.to_csv(SALIDA / nombre, index=False, encoding="utf-8-sig")
    print(f"{nombre:42s} {len(df):6d} filas × {df.shape[1]} columnas")
print("Guardado en:", SALIDA)

## 9 · Conclusiones y pendientes

In [ ]:
# Guarda: esta celda necesita la configuración (primera celda de código).
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

print("Años cubiertos")
print(f"  MapBiomas C4: {anios_mb[0]}–{anios_mb[-1]} ({len(anios_mb)} años, {len(anios_mb) - 1} periodos)")
print(f"  Landsat 8: {ANIOS_L8[0]}–{ANIOS_L8[-1]} (el último es parcial) | escenas en total: {len(escenas)}, útiles: {int(escenas['util'].sum())}")
print(f"  Intervalo más frecuente entre pasadas de una misma escena: {int(gaps.mode().iloc[0])} días")
print()
print(f"Bosque en Cusco según GRUPOS: {bosque_km2.loc[ultimo]:.1f} km² en {ultimo}; código 3 solo: "
      f"{float(tabla_area[3].loc[ultimo]) if 3 in tabla_area.columns else 0:.1f} km²")
print(f"Pérdida de bosque {anios_mb[0]}–{anios_mb[-1]}: {ind['pérdida de bosque (km²)'].sum():.2f} km² "
      f"| periodo máximo: {ind['periodo'][ind['pérdida de bosque (km²)'].idxmax()]} ({ind['pérdida de bosque (km²)'].max():.2f} km²)")
print(f"Expansión agrícola acumulada: {ind['expansión agrícola (km²)'].sum():.2f} km²")
print(f"Indicadores de MapBiomas en el CSV único: {len(mapbiomas_largo)} filas de {mapbiomas_largo['producto'].nunique()} producto(s) | "
      f"por resolución: {mapbiomas_largo.groupby('resolucion_temporal').size().to_dict()}")

**Decisiones que este notebook permite cerrar**
- Definición de bosque para Cusco (revisar `GRUPOS` con la sección 5.2).
- Regla de nubosidad para escena útil (`NUBES_MAX`).
- Si se suma Landsat 9 para tener una pasada cada 8 días (`INCLUIR_L9`).
- Índice espectral que usará LandTrendr (comparar NDVI, EVI, NBR y NDMI).

**Por confirmar en la documentación de MapBiomas:** ventana de meses del mosaico anual y clasificador de la Colección 4 (ATBD), y si existen assets de otros productos (fuego, agua) con resolución mensual: ver `productos_mapbiomas.csv`.

**Documentos que habrá que actualizar (no se tocaron)**
- `00_proyecto.md`: las dos líneas pasan a "imágenes Landsat 8 + métricas MapBiomas"; salen GEOBOSQUES y la comparación entre líneas.
- `01_alcance.md`: fuentes (solo Landsat 8 y MapBiomas), expansión agrícola calculable con MapBiomas, y periodo 2013 → hoy para Landsat y 2013–2025 para MapBiomas. Antes 2026 estaba fuera del periodo validado: hay que decidir si se mantiene así.
- `02_datos_fuentes.md`: quitar Sentinel-2, GEOBOSQUES, WorldCover, SRTM y OSM; añadir todas las escenas del año, la grilla común y la regla de nubosidad.
- `metodologia_general.md`: pasos 3, 4, 5 y 10 (preprocesamiento, umbral, muestras y validación) se reescriben.

**Pendiente de fondo:** la exactitud de MapBiomas medida contra sí mismo es circular; hace falta validación independiente (puntos revisados sobre imágenes).

## 10 · Descarga de imágenes Landsat 8, con su fecha
- **30 m nativos, EPSG:32719.** No se remuestrea a más resolución: Landsat 8 y MapBiomas son de 30 m.
- **Grilla común** de 30 m en UTM 19S, alineada con la grilla nativa de Landsat 8 cuando es posible, y fija para todos los archivos.
- **Dos descargas, cada una con su interruptor:**
  - `EXPORTAR_ESCENAS`: **todas** las escenas del catálogo (todo el año, 2013 → hoy), recortadas a la provincia, con 6 bandas de reflectancia escaladas (sin máscara de nubes) y `QA_PIXEL` para enmascarar después. Lo que la escena no cubre queda como NaN.
  - `EXPORTAR_COMPUESTOS`: un GeoTIFF por año con las 11 bandas de la sección 7.
- Ambos están en `False`: con `False` solo se imprime qué se descargaría y cuánto pesa. **Para descargar todo, pon ambos en `True`.**
- No se descargan mapas de MapBiomas.

**Cada imagen lleva su fecha de tres formas**
1. **En el nombre:** `escenas/AAAA/landsat8_AAAA-MM-DD_pPPPrRRR_<escena>.tif` y `landsat8_junago_AAAA.tif`.
2. **En los metadatos del GeoTIFF** (con `rasterio`): `DATE_ACQUIRED`, `SCENE_CENTER_TIME`, `SENSING_DATETIME_UTC`, `SCENE_ID`, `SATELLITE`, `WRS_PATH`, `WRS_ROW`, `CLOUD_COVER`, `CLOUD_COVER_LAND`, `PIXEL_CLEAN_PCT`, factor y offset de escala, y los nombres de banda.
3. **En el manifiesto** `manifiesto_descarga.csv`, con fecha y hora UTC de cada imagen.

La hora de la toma es UTC; en Lima (UTC−5) es cinco horas antes.

In [ ]:
# Guarda: esta celda necesita la configuración (primera celda de código).
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

EXPORTAR_ESCENAS = False
EXPORTAR_COMPUESTOS = False
GRID_CRS = "EPSG:32719"
SALIDA_L8 = RAIZ / "data" / "raw" / "landsat8"
SALIDA_ESC = SALIDA_L8 / "escenas"
DESC_ESC = list(BANDAS_L8.keys()) + ["QA_PIXEL"]
DESC_COMP = list(BANDAS_L8.keys()) + INDICES + ["n_obs"]

# Grilla nativa de Landsat 8 sobre la provincia (para alinear los píxeles)
esc0 = ee.Image(col_l8.filterDate(f"{ANIOS_L8[-2]}-01-01", f"{ANIOS_L8[-2] + 1}-01-01").first())
nativa = esc0.select("SR_B2").projection().getInfo()
print("Grilla nativa Landsat 8:", nativa["crs"], nativa["transform"])
print("Grilla nativa MapBiomas:", proj_mb.get("crs"), proj_mb.get("transform"))

if nativa["crs"] == GRID_CRS:
    off_x, off_y = nativa["transform"][2] % 30, nativa["transform"][5] % 30
    print("→ Landsat 8 está en", GRID_CRS, ": la grilla se alinea con sus píxeles.")
else:
    off_x, off_y = 0.0, 0.0
    print(f"⚠ Landsat 8 está en {nativa['crs']}, no en {GRID_CRS}: la grilla se ancla a múltiplos de 30 m y GEE remuestrea (vecino más cercano).")

xmin, ymin, xmax, ymax = prov.to_crs(epsg=32719).total_bounds
x0 = math.floor((xmin - off_x) / 30) * 30 + off_x
y0 = math.ceil((ymax - off_y) / 30) * 30 + off_y
GRID_TRANSFORM = [30, 0, x0, 0, -30, y0]
NCOLS = math.ceil((xmax - x0) / 30)
NFILAS = math.ceil((y0 - ymin) / 30)
print("Grilla de descarga:", GRID_CRS, GRID_TRANSFORM, f"→ {NCOLS} × {NFILAS} píxeles")

if proj_mb.get("crs") != GRID_CRS:
    print("Nota: MapBiomas está en", proj_mb.get("crs"), "; si más adelante se descargan sus mapas, "
          "se remuestrean a esta grilla con vecino más cercano y NO coincidirán 1 a 1 con su grilla nativa.")

In [ ]:
N_PIXELES = NCOLS * NFILAS

plan_esc = (escenas[["año", "escena", "fecha", "hora_centro", "inicio_utc", "path", "row", "nubes_%", "nubes_tierra_%", "limpio_%", "util", "satelite"]]
            .rename(columns={"nubes_%": "nubes_pct", "nubes_tierra_%": "nubes_tierra_pct", "limpio_%": "limpio_pct"}).copy())
plan_esc["fecha"] = plan_esc["fecha"].dt.strftime("%Y-%m-%d")
plan_esc["inicio_utc"] = plan_esc["inicio_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
plan_esc["archivo"] = ("escenas/" + plan_esc["año"].astype(str) + "/landsat8_" + plan_esc["fecha"]
                       + "_p" + plan_esc["path"].astype(int).astype(str).str.zfill(3)
                       + "r" + plan_esc["row"].astype(int).astype(str).str.zfill(3) + "_" + plan_esc["escena"] + ".tif")
plan_esc["bandas"] = len(DESC_ESC)
plan_esc["peso estimado sin comprimir (MB)"] = round(N_PIXELES * len(DESC_ESC) * 4 / 1e6, 1)
plan_esc["existe"] = [(SALIDA_L8 / a).exists() for a in plan_esc["archivo"]]

plan_comp = pd.DataFrame([{
    "archivo": f"landsat8_junago_{a}.tif", "año": a,
    "ventana": f"{a}-{FECHA_INI} → {a}-08-31", "escenas útiles jun–ago": utiles_jja.get(a, 0), "bandas": len(DESC_COMP),
    "peso estimado sin comprimir (MB)": round(N_PIXELES * len(DESC_COMP) * 4 / 1e6, 1),
    "existe": (SALIDA_L8 / f"landsat8_junago_{a}.tif").exists(),
} for a in anios_con_compuesto])

print(f"Compuestos: {len(plan_comp)} archivos, ~{plan_comp['peso estimado sin comprimir (MB)'].sum():.0f} MB sin comprimir")
print(f"Escenas:    {len(plan_esc)} archivos, ~{plan_esc['peso estimado sin comprimir (MB)'].sum() / 1000:.1f} GB sin comprimir")
plan_comp

In [ ]:
plan_esc[["año", "fecha", "hora_centro", "escena", "nubes_pct", "limpio_pct", "archivo", "peso estimado sin comprimir (MB)", "existe"]].head(10)

In [ ]:
def tags_escena(f):
    """Metadatos de fecha y calidad de una escena (fila de plan_esc), como diccionario de texto."""
    return {"DATE_ACQUIRED": f.fecha, "SCENE_CENTER_TIME": str(f.hora_centro), "SENSING_DATETIME_UTC": f.inicio_utc,
            "SCENE_ID": f.escena, "SATELLITE": f.satelite, "WRS_PATH": str(int(f.path)), "WRS_ROW": str(int(f.row)),
            "CLOUD_COVER": str(f.nubes_pct), "CLOUD_COVER_LAND": str(f.nubes_tierra_pct), "PIXEL_CLEAN_PCT": str(f.limpio_pct),
            "SCALE_FACTOR": "0.0000275", "SCALE_OFFSET": "-0.2", "BANDS": ",".join(DESC_ESC)}


def tags_compuesto(anio):
    """Metadatos de un compuesto anual: año, ventana, escenas usadas y regla de nubosidad."""
    usadas = escenas[(escenas["año"] == anio) & escenas["en_jun_ago"] & escenas["util"]]
    return {"YEAR": str(anio), "WINDOW": f"{anio}-{FECHA_INI} to {anio}-08-31", "COMPOSITE": "median",
            "N_SCENES": str(len(usadas)), "SCENE_IDS": ",".join(usadas["escena"]),
            "SCENE_DATES": ",".join(usadas["fecha"].dt.strftime("%Y-%m-%d")), "CLOUD_RULE": f"CLOUD_COVER<{NUBES_MAX}",
            "BANDS": ",".join(DESC_COMP)}


def escribir_metadatos(ruta, tags, descripciones):
    """Escribe tags y descripciones de banda en un GeoTIFF local. Devuelve False si falta rasterio."""
    try:
        import rasterio
    except ImportError:
        print("  falta rasterio (python -m pip install rasterio): no se escriben los metadatos de fecha")
        return False
    with rasterio.open(ruta, "r+") as dst:
        dst.update_tags(**tags)
        if len(descripciones) == dst.count:
            dst.descriptions = tuple(descripciones)
    return True


def descargar(img, ruta, dtype="float32"):
    """Descarga un GeoTIFF con geemap (por teselas). Si falla, lanza una tarea de exportación a Drive."""
    try:
        import geemap
        geemap.download_ee_image(img, str(ruta), region=aoi, crs=GRID_CRS, crs_transform=GRID_TRANSFORM,
                                 dtype=dtype, overwrite=True)
        return "descarga directa (geemap)"
    except Exception as e:
        print("  descarga directa falló:", repr(e)[:200])
        tarea = ee.batch.Export.image.toDrive(
            image=img, description=ruta.stem[:100], folder="cobertura_cusco", fileNamePrefix=ruta.stem,
            region=aoi, crs=GRID_CRS, crsTransform=GRID_TRANSFORM, maxPixels=1e10)
        tarea.start()
        return "tarea a Google Drive (carpeta cobertura_cusco)"


def imagen_escena(indice):
    """Escena sin máscara: 6 bandas de reflectancia escaladas + QA_PIXEL, recortada a la provincia."""
    img = ee.Image(f"{COLECCIONES[indice[:4]]}/{indice}")
    sr = img.select(list(BANDAS_L8.values()), list(BANDAS_L8.keys())).multiply(0.0000275).add(-0.2)
    return sr.addBands(img.select("QA_PIXEL")).toFloat().clip(aoi)


def ahora():
    return pd.Timestamp.now().isoformat(timespec="seconds")


def mb_de(ruta):
    return round(ruta.stat().st_size / 1e6, 1) if ruta.exists() else None


manifiesto = []
if EXPORTAR_COMPUESTOS:
    SALIDA_L8.mkdir(parents=True, exist_ok=True)
    for anio in anios_con_compuesto:
        ruta = SALIDA_L8 / f"landsat8_junago_{anio}.tif"
        if ruta.exists():
            print(anio, "compuesto ya existe, se salta")
            continue
        print(anio, "compuesto: descargando…")
        metodo = descargar(compuesto_anual(anio), ruta)
        if ruta.exists():
            escribir_metadatos(ruta, tags_compuesto(anio), DESC_COMP)
        manifiesto.append({"tipo": "compuesto", "archivo": ruta.name, "año": anio, "fecha": f"{anio}-{FECHA_INI} → {anio}-08-31",
                           "hora_utc": "", "escena": "", "path": "", "row": "", "nubes_pct": "", "limpio_pct": "",
                           "bandas": len(DESC_COMP), "crs": GRID_CRS, "pixel_m": 30, "metodo": metodo,
                           "mb": mb_de(ruta), "fecha_descarga": ahora()})
else:
    print("EXPORTAR_COMPUESTOS = False: no se descargaron compuestos.")

In [ ]:
if EXPORTAR_ESCENAS:
    for i, f in enumerate(plan_esc.itertuples(), start=1):
        ruta = SALIDA_L8 / f.archivo
        ruta.parent.mkdir(parents=True, exist_ok=True)
        if ruta.exists():
            continue
        print(f"[{i}/{len(plan_esc)}] {f.fecha} {f.hora_centro[:8]} UTC · {f.escena} · nubes {f.nubes_pct}%")
        metodo = descargar(imagen_escena(f.escena), ruta)
        if ruta.exists():
            escribir_metadatos(ruta, tags_escena(f), DESC_ESC)
        manifiesto.append({"tipo": "escena", "archivo": f.archivo, "año": f.año, "fecha": f.fecha, "hora_utc": f.inicio_utc,
                           "escena": f.escena, "path": int(f.path), "row": int(f.row), "nubes_pct": f.nubes_pct,
                           "limpio_pct": f.limpio_pct, "bandas": len(DESC_ESC), "crs": GRID_CRS, "pixel_m": 30,
                           "metodo": metodo, "mb": mb_de(ruta), "fecha_descarga": ahora()})
else:
    print("EXPORTAR_ESCENAS = False: no se descargaron escenas.")

if manifiesto:
    SALIDA_L8.mkdir(parents=True, exist_ok=True)
    ruta_man = SALIDA_L8 / "manifiesto_descarga.csv"
    nuevo = pd.DataFrame(manifiesto)
    if ruta_man.exists():
        nuevo = pd.concat([pd.read_csv(ruta_man, encoding="utf-8-sig"), nuevo], ignore_index=True)
    nuevo.to_csv(ruta_man, index=False, encoding="utf-8-sig")
    print("Manifiesto guardado:", ruta_man, "|", len(manifiesto), "archivos nuevos")

### 10.1 · Completar la fecha en los GeoTIFF que no la tengan
Útil para los archivos que lleguen desde Google Drive (la tarea de exportación no escribe metadatos). La celda es **idempotente**: solo toca los GeoTIFF locales a los que les falta `DATE_ACQUIRED` (escenas) o `YEAR` (compuestos).

In [ ]:
try:
    import rasterio
except ImportError:
    rasterio = None
    print("Falta rasterio: python -m pip install rasterio")

if rasterio is not None and SALIDA_L8.exists():
    por_id = {f.escena: f for f in plan_esc.itertuples()}
    completados = 0
    for ruta in sorted(SALIDA_ESC.glob("*/landsat8_*.tif")):
        with rasterio.open(ruta) as src:
            tiene = "DATE_ACQUIRED" in src.tags()
        if not tiene:
            escena = ruta.stem.split("_", 3)[-1]
            if escena in por_id and escribir_metadatos(ruta, tags_escena(por_id[escena]), DESC_ESC):
                completados += 1
    for ruta in sorted(SALIDA_L8.glob("landsat8_junago_*.tif")):
        with rasterio.open(ruta) as src:
            tiene = "YEAR" in src.tags()
        if not tiene and escribir_metadatos(ruta, tags_compuesto(int(ruta.stem[-4:])), DESC_COMP):
            completados += 1
    print("GeoTIFF a los que se les agregó la fecha:", completados)

In [ ]:
# Verificación de los GeoTIFF descargados (necesita rasterio)
if rasterio is not None:
    existentes = (sorted(SALIDA_L8.glob("landsat8_junago_*.tif"))[:1] + sorted(SALIDA_ESC.glob("*/landsat8_*.tif"))[:2]) if SALIDA_L8.exists() else []
    if not existentes:
        print("Todavía no hay GeoTIFF descargados.")
    for ruta in existentes:
        with rasterio.open(ruta) as src:
            datos = src.read(1)
            tags = src.tags()
            print(f"{ruta.name}: crs={src.crs} | píxel={src.res} | bandas={src.count} | tamaño={src.width}×{src.height} | "
                  f"% NaN banda 1={100 * np.isnan(datos).mean():.1f}")
            print("   fecha en metadatos:", tags.get("DATE_ACQUIRED") or tags.get("WINDOW") or "⚠ SIN FECHA",
                  "| bandas:", list(src.descriptions))